https://tianchi.aliyun.com/notebook/416271

In [1]:
import sys
from kaitoupao import *

storage dir: /Users/minkexiu/Downloads/GitHub/Tianchi_ChnAddrAnalysis
code dir: /Users/minkexiu/Documents/GitHub/Tianchi_ChnAddrAnalysis 

11 22 54
先天八卦数: 1乾, 2兑, 3离, 4震, 5巽, 6坎, 7艮, 8坤
本卦上：3 本卦下：6 变爻：6


,火水未济,水火既济,雷水解
上卦,☲离火,☵坎水,☳震木
下卦,☵坎水,☲离火,☵坎水


04 14 12 亥时
先天八卦数: 1乾, 2兑, 3离, 4震, 5巽, 6坎, 7艮, 8坤
本卦上：4 本卦下：6 变爻：6


,雷水解,水火既济,火水未济
上卦,☳震木,☵坎水,☲离火
下卦,☵坎水,☲离火,☵坎水


# 加载数据来看看

In [2]:
import codecs

In [3]:
# 读取训练集
train_lines = codecs.open(
    create_originalData_path("train.conll")
).readlines()

In [4]:
# 读取验证集
dev_lines = codecs.open(
    create_originalData_path("dev.conll")
).readlines()

In [5]:
dev_lines[0]

'杭 B-city\n'

In [6]:
# 读取测试集
test_lines = codecs.open(
    create_originalData_path("final_test.txt")
).readlines()

In [7]:
test_lines[0]

'1\x01朝阳区小关北里000-0号\n'

In [8]:
# 查看训练集中的前十个字符
# 这里左侧是输入的字符，右侧是字符对应的标签  
train_lines[:20]

['浙 B-prov\n',
 '江 E-prov\n',
 '杭 B-city\n',
 '州 I-city\n',
 '市 E-city\n',
 '江 B-district\n',
 '干 I-district\n',
 '区 E-district\n',
 '九 B-town\n',
 '堡 I-town\n',
 '镇 E-town\n',
 '三 B-community\n',
 '村 I-community\n',
 '村 E-community\n',
 '一 B-poi\n',
 '区 E-poi\n',
 '\n',
 '浙 B-prov\n',
 '江 I-prov\n',
 '省 E-prov\n']

In [9]:
# 接下来我们将拼接训练集，原始训练集是对单个字符
# train_str 为训练集字符串
# train_label 为训练集对应的标签
train_str, train_label = [''], [[]]

# 从训练集文本中依次读取文本
for line in train_lines:
    # 如果训练集行中存在空格，则表示格式为 【字符 标注】
    # 将当前的字符和标注累计
    if ' ' in line:
        word, label = line.strip().split(' ')
        train_str[-1] += word
        train_label[-1].append(label)
    
    # 否则就是一个句子结束，也就是'\n'的情况
    # 新增一个空的句子和标签
    else:
        train_str.append('')
        train_label.append([])

# 输出训练集第一个样本的文本和标签
train_str[0], train_label[0]

('浙江杭州市江干区九堡镇三村村一区',
 ['B-prov',
  'E-prov',
  'B-city',
  'I-city',
  'E-city',
  'B-district',
  'I-district',
  'E-district',
  'B-town',
  'I-town',
  'E-town',
  'B-community',
  'I-community',
  'E-community',
  'B-poi',
  'E-poi'])

In [10]:
# dev_str 为验证集字符串
# dev_label 为验证集对应的标签
dev_str, dev_label = [''], [[]]

# 按照训练集标签的格式，得到验证集输入的句子和对应的标签
for line in dev_lines:

    if ' ' in line:
        word, label = line.strip().split(' ')
        dev_str[-1] += word
        dev_label[-1].append(label)
    else:
        dev_str.append('')
        dev_label.append([])

# 输出验证集第一个样本的文本和标签
dev_str[0], dev_label[0]

('杭州五洲国际', ['B-city', 'E-city', 'B-poi', 'I-poi', 'I-poi', 'E-poi'])

# 数据分析

In [11]:
# 统计字符个数
len(set(''.join(train_str)))

2307

In [12]:
from scipy.stats import describe

# 统计输入文本的长度统计值
# 最长文本长度为68，最短是3，平均长度是17
describe([
    len(x) for x in train_str
])

DescribeResult(nobs=8856, minmax=(3, 69), mean=17.157859078590786, variance=32.80534308392209, skewness=1.0429906237323388, kurtosis=2.472641491590407)

In [13]:
# 将训练集标签和验证集标签进行拼接
label_set = sum(train_label, []) + sum(dev_label, [])

# 将标签进行去重
label_set = list(set(label_set))
# label_set.insert(0, 'O')
label_set

['B-poi',
 'B-floorno',
 'B-subpoi',
 'E-prov',
 'E-poi',
 'E-houseno',
 'I-devzone',
 'I-houseno',
 'E-district',
 'B-assist',
 'I-assist',
 'E-intersection',
 'E-village_group',
 'B-prov',
 'B-city',
 'B-town',
 'I-subpoi',
 'I-floorno',
 'I-prov',
 'B-distance',
 'B-roadno',
 'E-community',
 'B-road',
 'B-community',
 'E-roadno',
 'E-floorno',
 'S-community',
 'I-town',
 'S-district',
 'E-town',
 'I-city',
 'B-village_group',
 'E-cellno',
 'I-distance',
 'I-roadno',
 'I-cellno',
 'E-assist',
 'I-intersection',
 'S-assist',
 'I-community',
 'B-cellno',
 'B-houseno',
 'O',
 'E-subpoi',
 'E-distance',
 'I-village_group',
 'E-road',
 'B-intersection',
 'E-devzone',
 'E-city',
 'I-poi',
 'B-devzone',
 'S-intersection',
 'B-district',
 'I-road',
 'I-district',
 'S-poi']

In [14]:
from collections import Counter

# 利用counter统计标签的频次
c = Counter()
c.update(sum(train_label, []))

# 按照次序倒序进行展示
c.most_common()

[('I-poi', 16657),
 ('I-road', 8095),
 ('I-roadno', 7739),
 ('B-poi', 6354),
 ('E-poi', 6354),
 ('B-district', 6124),
 ('E-district', 6124),
 ('I-district', 5842),
 ('B-road', 5711),
 ('E-road', 5711),
 ('I-town', 5270),
 ('B-city', 5118),
 ('E-city', 5118),
 ('I-city', 4262),
 ('B-prov', 4003),
 ('E-prov', 4003),
 ('B-town', 3883),
 ('E-town', 3883),
 ('B-roadno', 3716),
 ('E-roadno', 3716),
 ('I-devzone', 3714),
 ('I-prov', 3688),
 ('O', 3588),
 ('I-houseno', 2335),
 ('B-houseno', 2187),
 ('E-houseno', 2187),
 ('I-subpoi', 2018),
 ('I-community', 1708),
 ('B-community', 1571),
 ('E-community', 1571),
 ('B-subpoi', 1437),
 ('E-subpoi', 1437),
 ('B-devzone', 1020),
 ('E-devzone', 1020),
 ('B-floorno', 827),
 ('E-floorno', 827),
 ('I-cellno', 491),
 ('B-cellno', 413),
 ('E-cellno', 413),
 ('I-floorno', 357),
 ('B-assist', 228),
 ('E-assist', 228),
 ('S-assist', 187),
 ('B-village_group', 152),
 ('E-village_group', 152),
 ('B-intersection', 103),
 ('E-intersection', 103),
 ('I-village_gr

In [15]:
# 上述是标签的统计结果，接下来我们对标签进行编码

# 从标签编码为数值
tag_to_ix = {key: idx for idx, key in enumerate(label_set)}

# 从数值还原为标签
ix_to_tag = {idx: key for idx, key in enumerate(label_set)}

tag_to_ix['E-assist'], ix_to_tag[40]

(36, 'B-cellno')

# 尝试1：LSTM实体识别

In [16]:
# 将序列进行编码
# seq 输入的序列
# to_ix 序列与标签的对应关系
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

In [17]:
# 训练数据，输入文本和标签，一一对应的关系
training_data = [[list(x), y] for x, y in zip(train_str, train_label)]

In [18]:
training_data[0]

[['浙',
  '江',
  '杭',
  '州',
  '市',
  '江',
  '干',
  '区',
  '九',
  '堡',
  '镇',
  '三',
  '村',
  '村',
  '一',
  '区'],
 ['B-prov',
  'E-prov',
  'B-city',
  'I-city',
  'E-city',
  'B-district',
  'I-district',
  'E-district',
  'B-town',
  'I-town',
  'E-town',
  'B-community',
  'I-community',
  'E-community',
  'B-poi',
  'E-poi']]

In [19]:
# 验证数据，输入文本和标签，一一对应的关系
dev_data = [[list(x), y] for x, y in zip(dev_str[:-1], dev_label[:-1])]

In [20]:
# 文本字符进行编码，得到字符与数值的对应的关系
word_to_ix = {}
for sent, tags in (training_data + dev_data):
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

for line in test_lines:
    line = line.strip()
    for word in line:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

# 搭建模型

In [21]:
# 导入pytorch，如果你能成功运行说明运行环境已经配置好了~
import torch
# 接下来导入pytorch功能模块
import torch.nn as nn # 用于搭建模型
import torch.optim as optim # 用于定义优化器

In [22]:
EMBEDDING_DIM = 128  # 文本嵌入的维度
HIDDEN_DIM = 32  # LSTM 隐含层维度

In [23]:
# 定义LSTM 实体抽取模型
class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        # 嵌入层，将文本进行嵌入
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # LSTM层，将文本提取特征
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # 分类层，将特征转换为类别
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
        
        # 将输出的概率进行归一化
        self.softmax = nn.Softmax(dim=1)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = self.softmax(tag_space)
        return tag_scores

In [24]:
# 定义模型
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
model

LSTMTagger(
  (word_embeddings): Embedding(3305, 128)
  (lstm): LSTM(128, 32)
  (hidden2tag): Linear(in_features=32, out_features=57, bias=True)
  (softmax): Softmax(dim=1)
)

# 模型训练

In [25]:
# 模型超参数
LR = 0.005 # 学习率
EPOCH = 3 # Epoch个数

In [26]:
# 定义损失函数
loss_function = nn.NLLLoss()

# 定义优化器
optimizer = optim.Adam(model.parameters(), lr=LR)

In [27]:
for epoch in range(EPOCH):
    model.train()
    # 模型训练部分
    for sentence, tags in tqdm.tqdm(training_data):
        # 模型梯度清空
        model.zero_grad()
        
        # 对输入的句子进行编码
        sentence_in = prepare_sequence(sentence, word_to_ix)

        # raise Exception("搁这儿停顿")
        
        # 对标签进行编码
        targets = prepare_sequence(tags, tag_to_ix)
        
        # 输入句子得到模型的输出
        tag_scores = model(sentence_in)
        
        # 计算模型输出与标签的概率差异性
        loss = loss_function(tag_scores, targets)
        
        # 通过损失计算模型参数的梯度
        loss.backward()
        
        # 模型的参数更新
        optimizer.step()

    # 验证部分
    val_acc = []
    model.eval()
    with torch.no_grad():
        for sentence, tags in tqdm.tqdm(dev_data[:]):
            # 对输入的句子进行编码
            sentence_in = prepare_sequence(sentence, word_to_ix)
            
            # 对标签进行编码
            targets = prepare_sequence(tags, tag_to_ix)
            
            # 输入句子得到模型的输出
            tag_scores = model(sentence_in)
            
            # 计算模型预测结果与真实标签的差异性，得到字符级别预测的准确率
            val_acc.append(
                (tag_scores.argmax(1) == targets).float().mean().item()
            )

    print('Epoch', epoch, np.mean(val_acc))

100%|████████████████████████████████████████████████████████████████████████████████████████| 1970/1970 [00:00<00:00, 6101.75it/s]


Epoch 0 0.5766447688419807


100%|████████████████████████████████████████████████████████████████████████████████████████| 1970/1970 [00:00<00:00, 6221.48it/s]


Epoch 1 0.5819776470096886


100%|████████████████████████████████████████████████████████████████████████████████████████| 1970/1970 [00:00<00:00, 6215.18it/s]

Epoch 2 0.5886150999268904


In [28]:
sentence_in

tensor([  0,   1,  13, 134,  76,   4, 134,  76,  17,  90, 644, 504, 970,  26,
         26,  26,  56])

# 预测打分

In [29]:
# 读取测试集文本进行预测
text = list(test_lines[25].strip().split('\x01')[1])

In [31]:
text

['北', '京', '市', '西', '城', '区', '二', '龙', '路', '0', '0', '号']

In [32]:
# 对标签进行编码
sentence_in = prepare_sequence(text, word_to_ix)

In [33]:
# 对测试集文本进行预测
tag_scores = model(sentence_in)

In [36]:
# 测试集预测结果
[ix_to_tag[x] for x in tag_scores.argmax(1).numpy()]

['B-road',
 'I-city',
 'E-city',
 'B-district',
 'I-district',
 'E-district',
 'B-poi',
 'I-poi',
 'E-road',
 'B-roadno',
 'I-roadno',
 'E-roadno']

In [37]:
## 批量预测

In [39]:
model.eval()
with open('lstm_addr_parsing_runid.txt', 'w', encoding='utf8') as up, torch.no_grad():
    for text in tqdm.tqdm(test_lines[:]):
        idx = text.strip().split('\x01')[0]
        text = text.strip().split('\x01')[1]

        sentence_in = prepare_sequence(text, word_to_ix)

        tag_scores = model(sentence_in)
        tag_prediction = [ix_to_tag[x] for x in tag_scores.argmax(1).numpy()]
        
        up.write(
            str(idx) + '\u0001' + text + '\u0001' + ' '.join(tag_prediction) + '\n'
        )

100%|██████████████████████████████████████████████████████████████████████████████████████| 50000/50000 [00:06<00:00, 7276.67it/s]
